In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd
csv_path = os.path.join(path, "Q3_data.csv")

og_df = pd.read_csv(csv_path)
df = og_df.copy()

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:

# removing columns with many missing data (drop cols with more than 2.5% of data missing)
cols = df.columns
for col in cols:
  if int(df[col].isnull().sum()) > 500:
    df = df.drop(columns=[col])




In [ ]:
# filling with median
cols = df.columns
columns_with_missing = []
for col in cols:
  if int(df[col].isnull().sum()) > 0:
    columns_with_missing.append(col)
    df[col] = df[col].fillna(df[col].median())



In [ ]:
df.isnull().sum()

In [ ]:
# Task 2: Write your code here:
# Control duplicates
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
# no categorical variables requiring encoding

print(df.select_dtypes(include=["object"]).columns)




In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()


cols = df.columns.drop("Target")


df[cols] = scaler.fit_transform(df[cols])



In [ ]:
# Task 5: Write your code here:
# Check balance
target_column = "Target"
print(df[target_column].value_counts())


#Target is imbalanced, we need to stratify our data & use metrics other than of accuracy

In [ ]:
# Task 1: Write your code here:
# Separate Features and Target

X = df.drop(target_column, axis=1)
y = df[target_column]

print("\nDataset Shapes")
print("X:", X.shape)
print("y:", y.shape)

In [ ]:
!pip install dask[dataframe] catboost

In [ ]:

import numpy as np
from sklearn.metrics import log_loss
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score, classification_report



In [ ]:
# Task 2,3,4,5: Write your code here:
model = CatBoostClassifier(verbose=0)

# Train model Stratified Kfold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_f1=[]

for train_index, test_index in skf.split(X, y):
      # Split data into training and testing sets
      X_Train, X_Test = X.loc[train_index, :], X.loc[test_index, :]
      y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
      # Train the model
      model.fit(X_Train, y_Train)
      # Predict on the test set
      y_pred = model.predict(X_Test)

      # Calculate metrics
      scores_f1.append(f1_score(y_Test, y_pred, average='weighted'))


    # Print the results
print(f" F1-Score: {np.mean(scores_f1):.4f}")
print("\n")



In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# Task 1: Write your code here:
# Plotting coefficients
importance = list(zip(X.columns, model.feature_importances_))
# importance = list(zip(X_scaled.columns, model.coef_[0])) try indexing the model.coef_ if you get an error stating numpy array should be 1D

sorted_importance = sorted(importance, key=lambda x: abs(x[1]), reverse=True)

# Extract sorted features and their coefficients
features, coefficients = zip(*sorted_importance)

# Plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(features, coefficients, color='darkblue')
plt.xlabel('Coefficient Value')
plt.ylabel('Features')
plt.title('Regression Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()


In [ ]:
# Task 2: Write your code here:
# the most important is D_45
golden_feature = sorted_importance[0][0]
print(golden_feature)


In [ ]:
# I STARTED ALL OVER AGAIN TO SOLVE THE BONUS BECAUSE IN THE FIRST PART I FILLED THE MISSING IN THE GOLDEN FEATURE WITH THE MEDIAN.
# Rather, i want to drop rows with missing data from the golden_feature

In [ ]:
# Task Bonus: Write your code here:
new_df = og_df.copy()

In [ ]:
new_df = new_df[[golden_feature,'Target']]

In [ ]:
#drop missing
new_df = new_df.dropna(subset=[golden_feature,'Target'])

In [ ]:
# Control duplicates
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(new_df)

In [ ]:
# Separate Features and Target
target_column = "Target"

new_X = new_df.drop(target_column, axis=1)
new_y = new_df[target_column]

print("\nDataset Shapes")
print("new_X:", X.shape)
print("new_y:", y.shape)

In [ ]:
# Feature Scaling
scaler = StandardScaler()
new_X = scaler.fit_transform(X)

In [ ]:
new_X = pd.DataFrame(new_X)

# Transfer back to dataframe (encoding returns numpy array if the input was series)
new_y = pd.DataFrame(new_y)

In [ ]:
# Task 2,3,4,5: Write your code here:
model = CatBoostClassifier(verbose=0)

# Train model Stratified Kfold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_f1=[]

for train_index, test_index in skf.split(new_X, new_y):
      # Split data into training and testing sets
      X_Train, X_Test = new_X.iloc[train_index, :], new_X.iloc[test_index, :]
      y_Train, y_Test = new_y.iloc[train_index], new_y.iloc[test_index]
      # Train the model
      model.fit(X_Train, y_Train)
      # Predict on the test set
      y_pred = model.predict(X_Test)

      # Calculate metrics
      scores_f1.append(f1_score(y_Test, y_pred, average='weighted'))


    # Print the results
print(f" F1-Score: {np.mean(scores_f1):.4f}")
print("\n")

